In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpretation-Conditioned Autoregressive Molecule Generation
Build base graphs under a fixed, immutable interpretation graph context.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
def estimated_predictive_performance(train_graphs, train_targets, test_graphs, test_targets):
    from nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
    train_X = vectorizer.transform(train_graphs)
    test_X = vectorizer.transform(test_graphs)
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    clf.fit(train_X, train_targets)
    from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
    y_prob = clf.predict_proba(test_X)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(test_targets, y_pred)
    print(f'Test accuracy: {acc:.3f} | ROC-AUC: {roc_auc_score(test_targets, y_prob):.3f} | AvgPrec: {average_precision_score(test_targets, y_prob):.3f}')

In [ ]:
def estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets):
    from nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    from sklearn.ensemble import ExtraTreesClassifier
    classifier = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)

    from abstractgraph_generative.generative_performance import compute_expected_gain_weighted_equivalent_data_size
    results = compute_expected_gain_weighted_equivalent_data_size(
        generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets,
        vectorizer=vectorizer, classifier=classifier,
        fracional_size=(1,2,3,4,5,7,10,15),
        n_repeats=30,
    )
    print(f'Expected gain weighted equivalent data size:{results["expected_gain_weighted_equivalent_data_size"]:.3f}')

    from abstractgraph_generative.generative_performance import plot_expected_gain_weighted_equivalent_data_size
    _ = plot_expected_gain_weighted_equivalent_data_size(results)

---

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[1]
assay_id = '624249' #bundled-safe assay example
size = int(2600/.33) #to account for 0.33 for test and 0.5 for reference
print(f"size: {size}")
use_equalized = True


limit_active = int(size // 2) if use_equalized else int(size)
limit_inactive = int(size // 2) if use_equalized else int(size)
graphs, targets = loader.load(
    assay_id,
    limit_active=limit_active,
    limit_inactive=limit_inactive,
)
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:
from sklearn.model_selection import train_test_split
test_size = 0.33
all_train_graphs, test_graphs, all_train_targets, test_targets = train_test_split(
    graphs,
    targets,
    test_size=test_size,
    stratify=targets if len(np.unique(targets)) > 1 else None,
    random_state=0,
)
reference_split_size = 0.5
train_graphs, reference_graphs, train_targets, reference_targets = train_test_split(
    all_train_graphs,
    all_train_targets,
    test_size=reference_split_size,
    stratify=all_train_targets if len(np.unique(all_train_targets)) > 1 else None,
    random_state=0,
)
print(f"AID {assay_id} graphs: {len(graphs)} (train={len(train_graphs)}, reference={len(reference_graphs)}, test={len(test_graphs)})")
#estimated_predictive_performance(train_graphs, train_targets, test_graphs, test_targets)

---

In [ ]:
from abstractgraph_generative.conditional_batch import ConditionalAutoregressiveGraphsGenerator

dataset_generator = ConditionalAutoregressiveGraphsGenerator(
    generator=generator,
    graph_estimator=graph_estimator,
    constraint_level=1,
    use_context_embedding=False,
    min_cluster_size=20,
    size_factor=1,
    verbose=True,
)

generated_graphs, generated_targets = dataset_generator.generate(
    train_graphs,
    train_targets,
)

estimated_generative_quality(
    generated_graphs,
    generated_targets,
    train_graphs,
    train_targets,
    reference_graphs,
    reference_targets,
    test_graphs,
    test_targets,
)


In [ ]:
from abstractgraph_generative.conditional_batch import ConditionalAutoregressiveGraphsGenerator

dataset_generator = ConditionalAutoregressiveGraphsGenerator(
    generator=generator,
    graph_estimator=graph_estimator,
    constraint_level=1,
    use_context_embedding=True,
    min_cluster_size=20,
    size_factor=1,
    verbose=True,
)

generated_graphs, generated_targets = dataset_generator.generate(
    train_graphs,
    train_targets,
)

estimated_generative_quality(
    generated_graphs,
    generated_targets,
    train_graphs,
    train_targets,
    reference_graphs,
    reference_targets,
    test_graphs,
    test_targets,
)


---